# Kalman Filter Price-Target Model — Fused Coregionalised Panel (v3)

**Notebook form of `pymc_kalman_filter_pt.py`** (0.9.9.11), aligned with
`probabilistic_ml_model/pymc_models/KalmanFilterModel.py`. Supersedes
`pymc_kalman_filter_pt_v2.ipynb`.

The cross-sectional spine is the **fused coregionalised (rank-1 ICM) panel model**
(`build_fused_kalman_pt_model`):

- **Model B spine** — a rank-1 Intrinsic Coregionalization Model (ICM) over the
  `(isin, time, y_series)` response tensor: response series share the latent per-ISIN
  factor `mu_isin` via per-series loadings (primary anchored at 1) with a per-series
  noise diagonal `sigma_series`. Per-time levels are **T direct per-time intercepts**
  (`alpha_level`) — the zero-anchored random-walk deviations were removed in the
  2026-08-01 reparameterisation after the T=4 validation isolated an aliased
  level/slope/innovation ridge (190 divergences → 0 after the fix); the time slope
  `beta_t` is materialised only when `t_scaled` genuinely varies across ISINs.
- **Model A refinement** — the risk-aware `expected_return → risk_adj_return` latent
  is the structural mean (`achieve_prob = sigmoid(risk_adj_return)`, a Deterministic),
  with the heteroscedastic scale `sigma_isin = sigma_base · (1 + cv) / √n` and the
  additive systematic-risk / size / volume tilts
  `risk_adj_return = ER − risk_loading·z(avg_beta) − size_loading·z(mcap_ratio) − volume_loading·z(rel_volume)`.

**What's new in v3** (see `CHANGELOG.md` 0.9.9.10 / 0.9.9.11):

1. **`KalmanRunConfig`** — every sampling / screen / risk-book / universe-query knob
   lives on one frozen dataclass (`from_env()` reads the environment), threaded
   through the whole workflow.
2. **Genuine (isin, time) T=4 panel — opt-in** — `history_lookbacks` builds a real
   log-uplift history panel from the `price_target_{6m,3m,1m}_ago` / `price_{…}_ago`
   trails (validated: 0 divergences, worst r_hat 1.00, ~16 min end-to-end).
3. **Return-space structural forecasts** — §10K/§11/§12 now plot observed vs
   **expected returns** per fiscal event (`plot_kalman_forecast_returns`, consuming
   the previously unused `implied_upside_future` forecast draws) instead of the
   price-target path.
4. **Standardised visualization suite** — semantic palette constants, hover
   templates, percent/probability axis formats, legend grouping, consolidated
   duplicate figures, plus new decision visuals (group-signal forest, CVaR-book
   composition, portfolio marker + efficient hull on the §10c dashboard).

## §0 — Environment, imports & run configuration

`PYTENSOR_FLAGS` must be set **before** PyTensor/PyMC are first imported (forward
slashes in the `cxx` path — the flag parser strips backslashes). Importing
`probabilistic_ml_model` normalises the flags; `set_env.ps1` handles the full setup.

All workflow knobs live on **`KalmanRunConfig`** (frozen dataclass):
`KalmanRunConfig.from_env()` resolves `RANDOM_SEED`, `KALMAN_PT_RESULTS_DIR`,
`KALMAN_PT_EXPORT_DRAWS`, `PML_FIG_WIDTH_PX` and `LOG_LEVEL`; everything else
(sampling budget, Monte-Carlo screen, CVaR book, universe-query dates) keeps its
dataclass default and is overridden with `dataclasses.replace`.

In [ ]:
import logging
import pymc as pm
from sqlalchemy import create_engine

# Section functions live in the module (single source of truth) — import, don't re-define.
import pymc_kalman_filter_pt as kf
from pymc_kalman_filter_pt import (
    KalmanRunConfig, set_run_config,
    setup_plotting, resolve_db_url,
    load_kalman_df, load_feature_catalogue, resolve_feature_roles,
    run_eda, map_state_space_features, prepare_kalman_panel_inputs,
    build_panel_model, run_prior_predictive, sample_posterior, run_posterior_predictive,
    run_diagnostics, summarize_panel_screen, compute_cvar_aware_book, export_analytics,
    run_single_isin_filter, run_single_isin_stochastic_vol,
    run_mingled_cohort_filter, run_mingled_cohort_stochastic_vol,
    run_granular_forest, run_granular_further_views,
    run_summary, run_recommendations,
    plot_screen_overview, plot_risk_return_scatter, plot_top_candidate_forest,
    present_group_effects, fit_kalman_model, KalmanFitResult,
    build_universe_consensus, report_universe_kalman_fit,
)

cfg = KalmanRunConfig.from_env()
logging.basicConfig(level=cfg.log_level)
setup_plotting()
engine = create_engine(resolve_db_url())
print('Setup complete — orchestrating:', kf.__file__)
cfg

### §0b — Optional: opt into the genuine T=4 history panel

The fused model runs on the collapsed **T=1** snapshot by default
(`panel_lookbacks=()`). The genuine `(isin, time)` panel — implied uplift at
6m/3m/1m ago plus the snapshot — is **validated** (2026-08-01 reparameterised run:
0 divergences, worst r_hat 1.00, worst bulk-ESS ≈ 1.6k, 15.7 min end-to-end) and
identifies drift betas the snapshot cannot (`feat_analyst_conviction` +0.022,
`feat_one_day_return` −0.024 — both null at T=1). Uncomment to opt in; the §4/§7
cells downstream honour whatever `cfg` carries.

In [ ]:
# cfg = dataclasses.replace(cfg, panel_lookbacks=('6m', '3m', '1m'))
set_run_config(cfg)
print('panel_lookbacks:', cfg.panel_lookbacks or '() — collapsed T=1 cross-section')

## §1 — Data load & feature-role resolution

- `load_kalman_df(engine, cfg)` — cross-sectional `pml.mv_pymc_kalman_pt` snapshot
  (one row per ISIN). The universe window (`next_earnings >= cfg.min_next_earnings`,
  `income_statement_report_date >= cfg.min_report_date`) is **config-driven** — the
  former hardcoded date literals are gone; roll them forward on `KalmanRunConfig`.
- `load_feature_catalogue` — the `kalman_pt` rows of `pml.vw_pymc_feature_catalogue`
  (the SQL registry SSOT).
- `resolve_feature_roles` — groups columns by `pymc_role` (catalogue SSOT, MV-schema
  fallback).

In [ ]:
kalman_df = load_kalman_df(engine, cfg)
feature_catalogue = load_feature_catalogue(engine)
roles = resolve_feature_roles(kalman_df, feature_catalogue)
kalman_df.head()

## §2 — Exploratory data analysis

`run_eda` renders the EDA panels through a state-space lens: drift features → the
state-transition mean (`beta` slopes), noise wideners → the measurement-noise scale.
Since 0.9.9.11 the panels carry decision context by default:

- the **industry ridge** is sorted by median with a 0% reference line;
- the **driver facets** annotate each facet with its Spearman ρ (plus an OLS trend
  line when `statsmodels` is available), so signal strength reads off directly;
- the former seven per-coord group forests are **consolidated into one faceted
  panel** gated to the coords the fused model actually uses as group effects,
  levels sorted by median with universe-median and 0% reference lines.

In [ ]:
run_eda(kalman_df, roles)

## §3 — State-space feature mapping

`map_state_space_features` maps the catalogue's `kalman_pt` mutable_predictors onto
Kalman roles via `KalmanFilterPriceTarget.select_drift_features` — the SSOT partition
(leakage, noise wideners, named tilts, support counters, rating counts, the collinear
composition leg, Piotroski components, `days_*` time covariates all stay out of the
drift matrix). Returns the drift-feature list + a tidy role-mapping frame.

> **Posterior note.** On the T=1 snapshot `beta[feat_one_day_return]` and
> `beta[feat_analyst_conviction]` straddle zero; on the genuine T=4 panel both are
> clearly identified (+0.022 / −0.024, 89% ETIs excluding 0) — an argument for the
> §0b opt-in, and the reason both stay in the drift matrix.

In [ ]:
drift_features, mapping = map_state_space_features(kalman_df, feature_catalogue)
mapping

## §4 — Fused-panel data containers

`prepare_kalman_panel_inputs` filters to log-space-usable rows and builds the
`KalmanPanelInputs`: the standardised `(isin, time, y_series)` response tensor `Y`,
the standardised time matrix `t_scaled`, the drift design matrix, the noise-widener
drivers, the systematic-risk / size / volume tilt inputs, and the categorical group
coords.

**Time axis.** With `history_lookbacks=()` (default) the panel is the collapsed
T=1 cross-section and `t_scaled` is the standardised days-to-earnings covariate.
With lookbacks (e.g. `('6m', '3m', '1m')`) each lookback's implied uplift
`price_target_{lb}_ago / price_{lb}_ago − 1` is winsorised to [−95%, +500%] and
`log1p`-mapped onto the response scale — a **genuine** oldest→newest history panel
(T = len + 1) with the snapshot as the final step; missing history cells (~1.5%)
are filled with the name's **own** snapshot uplift, never a cross-sectional-mean
fake observation. The old `collapse_time=False` branch that merely tiled the
snapshot across fiscal anchors (T-fold double-counting) is gone.

> **Primary response = log uplift.** `feat_log_uplift = log1p(feat_implied_upside)`
> — modelling the log keeps `expected_pt = last_price · exp(log_uplift)` strictly
> positive. `feat_implied_upside` itself is leakage-barred from the drift matrix.

In [ ]:
panel = prepare_kalman_panel_inputs(kalman_df, roles, drift_features,
                                    history_lookbacks=cfg.panel_lookbacks)
print('Y shape (isin, time, y_series):', panel.Y.shape)
print('response_names:', panel.response_names)
print('drift_names   :', panel.drift_names)

## §5 — Build the fused coregionalised model

`build_panel_model` wraps `build_fused_kalman_pt_model` and renders the model graph.

### Generative form (post-reparameterisation)

Per ISIN $i$, time step $t$, response series $d$ (primary $d{=}0$ is
`feat_log_uplift`):

**Model A — risk-conditioned drift baseline.** A hierarchical regression on the
standardised drift design with crossed, fixed-scale sum-to-zero group intercepts:

$$\eta_i = X^{\text{drift}}_i\,\beta + \sum_g \big(e^{(g)}\big)_{[i]},\qquad
\beta \sim \mathcal{N}(0,1),\ \ e^{(g)} \sim \text{ZSN}(\sigma{=}0.25)$$

$$\text{risk\_adj\_return}_i = \eta_i - \lambda\,z(\bar\beta_i)
- \gamma\,z(\text{mcap ratio}_i) - \kappa\,z(\text{rel\_volume}_i),\qquad
\text{achieve\_prob}_i = \sigma(\text{risk\_adj\_return}_i)$$

**Model B — rank-1 ICM spine.**

$$\mu^{\text{reg}}_{i,t,d} = \alpha_{t,d} + \beta^{(t)}_d\, t_{i,t}
+ W_d\,\mu^{\text{isin}}_i,\qquad
y_{i,t,d} \sim \text{StudentT}\big(\nu,\ \mu^{\text{reg}}_{i,t,d},\
\sigma^{\text{isin}}_i \tau_d\big)$$

- $\alpha_{t,d}$ are **T direct per-time intercepts** — exactly identified. The
  former zero-anchored GRW deviations (+ global slope) were mutually aliased per
  time slice on an isin-constant lookback axis and reproduced the historic
  scale×innovation ridge (2026-08-01 T=4 run: 190 divergences, `alpha_level` r_hat
  1.06 — fixed by this reparameterisation; re-validation: **0 divergences**).
- $\beta^{(t)}_d$ exists only when $t$ varies across ISINs (the T=1 days-to-event
  covariate); on an isin-constant axis it is fixed at 0.
- $W_d$ is the sign-fixed coregion loading (primary ≡ 1); $\tau_d$ the per-series
  noise diagonal; $\nu \ge 2.5$ the tail dof.

In [ ]:
robust = False
volume_penalty = 0.25  # HalfNormal prior scale for the learned volume_loading tilt; 0.0 disables it
model = build_panel_model(panel, robust=robust, volume_penalty=volume_penalty)
pm.model_to_graphviz(model)

## §6 — Prior predictive checks

`run_prior_predictive` draws `cfg.prior_draws` prior samples of `expected_return`,
`risk_adj_return`, `achieve_prob` and `sigma_isin` and renders three panels. Since
0.9.9.11 the implied-upside panel is on the **percent** scale (the module-wide
convention; it was the sole decimal-axis figure) with probability axes formatted
as percentages.

In [ ]:
prior_idata = run_prior_predictive(model, panel, cfg)
prior_idata.prior

## §7 — Posterior inference (NUTS)

`sample_posterior` tries `nutpie → numpyro → pymc` in priority order and merges the
prior groups into the posterior `DataTree`. The budget comes from the config:
`draws/tune/chains/target_accept/random_seed` (defaults 1000/1000/4/0.9/42).

> **cores=1 in the IDE kernel.** Launching nutpie's parallel native workers inside
> an IDE-managed Jupyter kernel on Windows can crash the kernel process (an
> uncatchable native crash). Chains run sequentially here; the standalone script
> path uses `cfg.cores` (typically 4).

**Validation history** (why the settings look like this):

| run                                    | geometry                             | result                                            |
|----------------------------------------|--------------------------------------|---------------------------------------------------|
| T=1 baseline                           | direct intercepts, no time axis      | 0 div, r_hat ≤ 1.01, ESS > 400                    |
| T=4 (2026-07-31/08-01, GRW deviations) | aliased level/slope/innovation block | 315 / 190 divergences, `alpha_level` r_hat 1.06   |
| T=4 reparameterised (2026-08-01)       | per-time direct intercepts           | **0 div, worst r_hat 1.00, ESS ≈ 1.6k, 15.7 min** |

No tune bump is needed for T > 1 since the reparameterisation — the panel shares
the T=1 geometry.

In [ ]:
idata = sample_posterior(model, prior_idata, cores=1, config=cfg)
print('Group effects fitted:', present_group_effects(idata))
idata.posterior

## §8 — Posterior predictive checks

Calibration of the standardised `(isin, time, y_series)` likelihood, consolidated
(0.9.9.11) to one ECDF overlay + the t-stat checks + PIT:

- **Pooled ECDF overlay** — replicate ECDFs vs the observed ECDF.
- **t-stat calibration** (`mean`, `std`) — observed T(y) inside the replicated
  distribution.
- **94% coverage** — printed per series AND charted against the 0.94 target line.
- **PIT ECDF** — uniform / in-band when calibrated.

In [ ]:
run_posterior_predictive(model, idata, panel)

## §9 — MCMC diagnostics

`run_diagnostics` reports R-hat / bulk-&-tail ESS, divergences, trace / rank-dist /
forest views over the fused hyper-parameters (`FUSED_SCALAR_VARS`, per-coord
`sigma_<coord>`, drift `beta`, the per-series coregion terms), the NUTS energy,
prior→posterior contraction and ESS evolution. Gates follow Vehtari et al. (2021):
**R-hat < 1.01**, **ESS > 400**. The variance partition is printed and (0.9.9.11)
rendered as a stacked share bar. Note: `sigma_alpha_innov` / `sigma_beta_innov` no
longer exist — the per-time direct intercepts removed them.

In [ ]:
run_diagnostics(idata, panel)

## §10 — Expected price targets: posterior screen

`summarize_panel_screen` → a `ScreenContext` with the posterior `expected_upside` /
`expected_pt` draws, the per-ISIN screening `results` table and the structural-TS
Monte-Carlo summary (`er_*` columns are genuine **decimal returns**; percent
scaling happens only at display boundaries). The Monte-Carlo horizon/damping come
from `cfg.mc_horizon` / `cfg.mc_rho`.

Renders: the per-industry posterior forest (0-line, sorted), the fused-model
internals panel, and the comparative-returns views (percent-space shrinkage
scatter, distributional KDE overlay, per-sector forest). The former price-space
shrinkage scatter and duplicate average-upside KDE were consolidated away.

In [ ]:
screen = summarize_panel_screen(idata, panel, horizon=cfg.mc_horizon, rho=cfg.mc_rho)
results = screen.results
results.head(15)

### §10b — CVaR-aware risk analytics & sizing (RiskBook)

Single source of truth for the risk layer: per-name expected shortfall (CVaR of
the posterior upside draws), a reward-to-CVaR (STARR) ranking, and a
per-name-capped long book with joint-draw portfolio aggregates. Knobs from the
config: `cvar_alpha` / `weight_cap` / `k_book` / `p_long`. The same `RiskBook`
feeds the §10c export and the §14b recommendations.

In [ ]:
risk_book = compute_cvar_aware_book(idata, panel, screen, results, config=cfg)
risk_book.book.head(25)

### §10c — Analytics export & decision dashboard

`export_analytics` maps the posterior onto `analytics.kalman_filtered_price_targets`
(decimal-return convention; unit changes ship with a GEIB dashboard deploy). The
overview dashboard now carries the **portfolio star** (aggregate E[r]/vol/CVaR from
`RiskBook.summary`), the **held-name efficient hull** on the risk-return map, and
probability axes as percentages; the shrinkage view reuses the shared
`create_kalman_vs_raw_scatter` (signed-log axes). Set `write=True` to persist.

In [ ]:
kalman_results = export_analytics(idata, panel, screen, risk_book=risk_book, write=True)
kalman_results.head()

### §10K — Universe-consensus fit (`fit_kalman_model` on the full universe)

`build_universe_consensus` pools every row's `price_target*_ago` trail into one
weekly-median consensus series; `fit_kalman_model` fits the funnel-free
marginalized GRW (+trend), spot-anchored at the universe-median `last_price`.
Since 0.9.9.11 the structural forecast renders in **return space** — observed vs
expected returns per fiscal event with a 0% break-even line — and the forecast
table always carries `implied_upside_pct`.

In [ ]:
universe = build_universe_consensus(kalman_df)
assert universe is not None, 'kalman_df carries no price_target*_ago history'
res_10k: KalmanFitResult = fit_kalman_model(
    price_targets=universe.observed, isin=universe.label, dates=universe.dates,
    last_price=universe.last_price, samples=2500, tune=2500, chains=4,
    target_accept=0.9, random_seed=cfg.random_seed, trend=True,
    parameterization='marginalized', nuts_sampler='nutpie',
    forecast_df=kalman_df, forecast_aggregate='median',
    forecast_anchor_col='fy_end_date')
report_universe_kalman_fit(res_10k, universe)

## §11 — Single-ISIN time-series Kalman filter (+ §11b stochastic volatility)

The literal single-security GRW filter on the richest `*_ago` history (time axis
anchored on `income_statement_report_date`). The §11 candidate pull is
config-driven (`min_mcap_country_rank`, `candidate_limit`). The structural
forecast is the **return-space** panel (`plot_kalman_forecast_returns`): observed
implied returns, the smoothed implied-upside band, nested latent/predictive
forecast bands per fiscal event (the gap between them is the analyst observation
noise), per-horizon +X% annotations. §11b refits with stochastic volatility — its
σ_obs(t) path renders as the companion row (posterior median).

In [ ]:
single_ctx = run_single_isin_filter(panel.frame, engine, cfg)
run_single_isin_stochastic_vol(single_ctx)

## §12 — Mingled-ISIN earnings-window cohort filter (+ §12b stochastic volatility)

Every ISIN whose `next_earnings` lands within ±`cfg.earnings_window_days` of today
is unpivoted and the cross-sectional **median** target taken per shared as-of date —
one earnings-cohort consensus series, fit with the marginalized GRW (+trend).
Return-space forecast + upside-bearing forecast table, as in §11.

In [ ]:
mingled_ctx = run_mingled_cohort_filter(panel.frame, engine, cfg)
run_mingled_cohort_stochastic_vol(panel.frame, mingled_ctx)

## §13 — Granular earnings-cohort posterior-predictive forest (+ §13.1 further views)

Keeps the §12 cohort definition but stays per-ISIN granular, reusing the fitted
fused posterior (no refit): per-name `expected_pt` posterior forests with the raw
analyst targets overlaid and pooled reference bands.

In [ ]:
forest_ctx = run_granular_forest(idata, results, panel, screen, engine, cfg)
run_granular_further_views(prior_idata, panel, screen, forest_ctx)

## §14 — Comprehensive summary & actionable recommendations

`run_summary` consolidates the run into an earnings-cohort vs baseline vs universe
read — since 0.9.9.11 the cross-sectional table and the sector mix are also
rendered as decision panels (grouped metric bars; cohort-vs-universe sector-tilt
diverging bar).

`run_recommendations` (§14b) turns the posterior into risk-aware signals. The
group-allocation block renders the **shrunk-excess forest** (per-coord OW/UW bands,
verdict-coloured); the CVaR sizing block renders the **book composition** chart
with the portfolio aggregates (E[upside] / CVaR5 / reward-to-CVaR /
diversification) that previously lived only in the prints.

In [ ]:
run_summary(results, screen, forest_ctx, mingled_ctx)
run_recommendations(idata, panel, results, screen, forest_ctx, risk_book=risk_book)

### §14.1 — Screen overview, risk/return screen & top-candidate forest

The former notebook-inline plotting functions are promoted into the module
(0.9.9.11) — import, don't re-define:

- `plot_screen_overview` — upside distribution + top-N ranked names with HDIs;
- `plot_risk_return_scatter` — interactive upside vs posterior-uncertainty screen
  (colour = sector, size = market cap);
- `plot_top_candidate_forest` — posterior expected-upside forest of the top names.

In [ ]:
plot_screen_overview(results, top_n=50)
plot_risk_return_scatter(results)
plot_top_candidate_forest(screen, results, top_n=50)

---
### Artifacts available at the top level

`cfg`, `kalman_df`, `roles`, `drift_features`, `panel`, `model`, `prior_idata`,
`idata`, `screen`, `results`, `risk_book`, `kalman_results`, `universe`, `res_10k`,
`single_ctx`, `mingled_ctx`, `forest_ctx`.

To run everything in one shot instead:
`kf.main(run_eda_section=True, write_analytics=False, export_results=True, config=cfg)`
— artifacts (PNG/CSV/JSON/NetCDF) land under `KALMAN_PT_RESULTS_DIR`.